In [ ]:
# Cell 1: Clone or pull the private repo using a PAT entered via getpass
# (the token is never hardcoded or printed/logged), then purge any
# already-imported src.* modules so a stale cached module from an earlier
# cell run in this session can never silently run instead of the code just
# pulled, and print the commit actually checked out. Writes to
# /kaggle/working -- the only writable location on a Kaggle notebook
# instance; /kaggle/input (where the datasets live) is read-only.
import os
import subprocess
import sys
from getpass import getpass

REPO_DIR = "/kaggle/working/plantguard-v2"
pat = getpass("GitHub Personal Access Token: ")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    remote = f"https://{pat}@github.com/RUDRAIndia/plantguard-v2.git"
    subprocess.run(["git", "clone", remote, REPO_DIR], check=True)

del pat
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

purged = sorted(name for name in sys.modules if name.startswith("src"))
for name in purged:
    del sys.modules[name]
print(f"Purged {len(purged)} cached src module(s): {purged}")

commit_hash = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"Checked out commit {commit_hash}")


In [ ]:
# Cell 2: Verify GPU visibility. The whole reason for moving training to
# Kaggle is its two attached Tesla T4s -- fail loudly here rather than
# silently falling back to a CPU-only run that would go unnoticed for hours
# (CLAUDE.md rule 1). Also confirms /kaggle/input is actually mounted, which
# src/config.py's IS_KAGGLE detection depends on.
from pathlib import Path

import tensorflow as tf

if not Path("/kaggle/input").is_dir():
    raise RuntimeError(
        "/kaggle/input does not exist -- this does not look like a Kaggle "
        "notebook instance. src/config.py's IS_KAGGLE detection depends on "
        "it."
    )

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__}")
print(f"GPUs visible: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

if not gpus:
    raise RuntimeError(
        "No GPU visible to TensorFlow. Check the notebook's Settings -> "
        "Accelerator is set to a GPU option before re-running -- training "
        "on CPU here would be far too slow to be usable."
    )


In [ ]:
# Cell 3: Validate the mounted dataset inputs in place. On Kaggle,
# PlantVillage and the negatives source are read-only notebook inputs,
# already extracted -- src/config.IS_KAGGLE routes both functions below to
# validate the mount directly (38-class assertion + image-count range,
# never weakened) and record its provenance, instead of downloading
# anything. A validation failure names exactly which input is wrong and
# that it needs re-attaching -- see src/data/download.py and
# src/data/negatives.py for the Kaggle-branch detail.
from src.data import download, negatives

download.download_plantvillage()
negatives.download_negatives()


In [ ]:
# Cell 4: Run the training smoke test for one model end to end (frozen
# head phase + fine-tune phase, on a small synthetic image set), then print
# the resulting history JSON -- verifies the whole training path on Kaggle
# before committing to a real run. Same code path as
# colab/01_data_setup.ipynb's Cell 11.
import json

from src import config, train

manifest = train.run_training(model_name=config.CANDIDATE_MODELS[0], smoke=True)
print(json.dumps(manifest, indent=2))

history_path = config.ARTIFACTS_DIR / f"history_{config.CANDIDATE_MODELS[0]}.json"
print(history_path.read_text(encoding="utf-8"))
